# RepairWise Gemma — Ollama Local Demo
## Gemma 4 Good Hackathon · Special Technology Prize Track (Ollama)

Run RepairWise with **Gemma 4 E2B via Ollama** — fully local, no GPU required for inference,
no HuggingFace token needed after model download.

This notebook demonstrates:
- Installing and running Ollama in Kaggle
- Pulling `gemma4:e2b` model
- Running RepairWise multilingual phone repair assistant via Ollama
- Streaming responses token by token
- Multimodal photo analysis (SMS screenshots, physical damage)

### Why Ollama?
Ollama makes Gemma 4 accessible to anyone with a laptop — no Python GPU stack,
no CUDA drivers. A phone repair shop owner can run RepairWise locally with:
```bash
ollama run gemma4:e2b
python app.py
```


## 1. Install Ollama

In [ ]:
import subprocess, sys, os, time

# Install Ollama (Linux)
print("Installing Ollama...")
result = subprocess.run(
    "curl -fsSL https://ollama.ai/install.sh | sh",
    shell=True, capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-200:] if result.stderr else "")
print("✅ Ollama installed")


## 2. Start Ollama server and pull Gemma 4 E2B

In [ ]:
import subprocess, time, urllib.request, json

# Start Ollama server in background
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

# Verify server is running
try:
    resp = urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5)
    print("✅ Ollama server running")
except Exception as e:
    print(f"⚠️  Server may still be starting: {e}")

# Pull Gemma 4 E2B
print("Pulling gemma4:e2b (this may take a few minutes)...")
result = subprocess.run(
    ["ollama", "pull", "gemma4:e2b"],
    capture_output=True, text=True, timeout=600
)
print(result.stdout[-300:] if result.stdout else "")
print("✅ gemma4:e2b ready")


## 3. Test raw Ollama connection

In [ ]:
import urllib.request, json

def ollama_generate(prompt: str, model: str = "gemma4:e2b", max_tokens: int = 200) -> str:
    payload = json.dumps({
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": max_tokens,
            "temperature": 0.4,
            "repeat_penalty": 1.12,
            "stop": ["<end_of_turn>", "<start_of_turn>"],
        }
    }).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read())
        return data.get("response", "").strip()

# Quick sanity check
response = ollama_generate("Say hello in Spanish in one sentence.")
print("Ollama response:", response)


## 4. Load RepairWise with Ollama backend

In [ ]:
import os
import sys

# Point to our RepairWise files (copy from /kaggle/input or define inline)
os.environ["REPAIRWISE_BACKEND"] = "ollama"
os.environ["OLLAMA_URL"] = "http://localhost:11434"
os.environ["OLLAMA_MODEL"] = "gemma4:e2b"

# If running from Kaggle with our dataset attached:
# sys.path.insert(0, "/kaggle/input/repairwise-gemma")

# For this notebook we inline the core functions
# In production: from engine import repairwise_debug

print("✅ Environment configured for Ollama backend")
print(f"  REPAIRWISE_BACKEND: {os.environ['REPAIRWISE_BACKEND']}")
print(f"  OLLAMA_URL: {os.environ['OLLAMA_URL']}")
print(f"  OLLAMA_MODEL: {os.environ['OLLAMA_MODEL']}")


## 5. RepairWise Ollama — multilingual demo

In [ ]:
import urllib.request, json, re, unicodedata

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "gemma4:e2b"

SYSTEM_PROMPT = (
    "You are RepairWise Gemma, a professional phone repair technician and digital safety advisor. "
    "Help people — especially elderly users and immigrants — understand phone problems and avoid SMS scams. "
    "Always respond in the SAME language as the user. "
    "Use EXACTLY this 5-section structure:\n"
    "1. Probable diagnosis:\n"
    "2. Risk level: HIGH 🔴 / MEDIUM 🟡 / LOW 🟢\n"
    "3. What to do right now:\n"
    "4. When to see a professional:\n"
    "5. Sources used: [knowledge_id]"
)

def repairwise_ollama(user_query: str, language: str = "Spanish") -> str:
    prompt = (
        f"<start_of_turn>user\n"
        f"{SYSTEM_PROMPT}\n\n"
        f"Language: {language}\n"
        f"Customer query: {user_query}\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 300,
            "temperature": 0.4,
            "repeat_penalty": 1.12,
            "stop": ["<end_of_turn>", "<start_of_turn>"],
        }
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=90) as resp:
        data = json.loads(resp.read())
        return data.get("response", "").replace("<end_of_turn>","").strip()

# ── Test cases ────────────────────────────────────────────────────────────────
test_cases = [
    ("Me ha llegado un SMS del banco con un link y me pide la tarjeta.", "Spanish"),
    ("Mi batería está hinchada y la pantalla se está levantando.", "Spanish"),
    ("I dropped my phone in water and now it is getting hot.", "English"),
    ("El meu mòbil no s'encén i es queda al logo.", "Catalan"),
    ("Telefonul meu nu se încarcă deloc.", "Romanian"),
    ("مجھے بینک کا ایک پیغام آیا ہے جس میں کارڈ کی تفصیلات مانگی گئی ہیں۔", "Urdu"),
]

print("=" * 70)
print("RepairWise Gemma — Ollama Backend Demo")
print(f"Model: {OLLAMA_MODEL} | Backend: Ollama local")
print("=" * 70)

for query, lang in test_cases:
    print(f"\n[{lang}] {query[:60]}...")
    try:
        answer = repairwise_ollama(query, lang)
        print(answer[:400])
    except Exception as e:
        print(f"Error: {e}")
    print("-" * 70)


## 6. Streaming demo — token by token

In [ ]:
import urllib.request, json

def repairwise_ollama_stream(user_query: str, language: str = "Spanish"):
    """Stream RepairWise response token by token via Ollama."""
    prompt = (
        f"<start_of_turn>user\n"
        f"{SYSTEM_PROMPT}\n\nLanguage: {language}\nCustomer query: {user_query}\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": True,
        "options": {"num_predict": 300, "temperature": 0.4, "repeat_penalty": 1.12,
                    "stop": ["<end_of_turn>", "<start_of_turn>"]},
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    full = ""
    with urllib.request.urlopen(req, timeout=90) as resp:
        for line in resp:
            if line:
                chunk = json.loads(line)
                token = chunk.get("response", "")
                print(token, end="", flush=True)
                full += token
                if chunk.get("done"):
                    break
    print()
    return full

print("Streaming demo — Mi batería está hinchada:")
print("-" * 70)
result = repairwise_ollama_stream(
    "Mi batería está hinchada y la pantalla se levanta.",
    language="Spanish"
)


## 7. Multimodal — photo analysis via Ollama

In [ ]:
import urllib.request, json, base64
from PIL import Image, ImageDraw
from io import BytesIO

def create_sms_screenshot():
    """Create a synthetic phishing SMS screenshot for demo."""
    img = Image.new("RGB", (600, 280), "white")
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, 600, 50], fill="#CC0000")
    d.text((15, 12), "⚠️  BANCO SEGURO — ALERTA", fill="white")
    d.text((15, 70), "Su cuenta ha sido SUSPENDIDA por actividad inusual.", fill="black")
    d.text((15, 100), "Verifique AHORA: http://banco-seguro-verificar.example/login", fill="#CC0000")
    d.text((15, 130), "Introduzca: número de tarjeta + PIN + código OTP", fill="#880000")
    d.text((15, 180), "⚠️ AVISO: Los bancos reales NUNCA piden esto por SMS.", fill="#007700")
    return img

def analyse_photo_ollama(image: Image.Image, query: str, language: str = "Spanish") -> str:
    """Send image + text to Ollama for multimodal analysis."""
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    img_b64 = base64.b64encode(buf.getvalue()).decode()

    prompt = (
        f"<start_of_turn>user\n"
        f"{SYSTEM_PROMPT}\n\nLanguage: {language}\n"
        f"The user has uploaded an image. Analyse it carefully.\n"
        f"Customer query: {query}\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "images": [img_b64],
        "stream": False,
        "options": {"num_predict": 300, "temperature": 0.4, "repeat_penalty": 1.12,
                    "stop": ["<end_of_turn>", "<start_of_turn>"]},
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=90) as resp:
        data = json.loads(resp.read())
        return data.get("response", "").replace("<end_of_turn>","").strip()

# Demo
sms_img = create_sms_screenshot()
print("Analysing SMS screenshot with Ollama vision...")
print("-" * 70)
answer = analyse_photo_ollama(sms_img, "¿Es real este mensaje del banco?", "Spanish")
print(answer)


## 8. Architecture summary

```
User (local machine)
      │
      │  ollama run gemma4:e2b
      ▼
┌──────────────────────┐
│  Ollama Server       │  http://localhost:11434
│  gemma4:e2b          │  Text + Vision
│  Runs on CPU or GPU  │  No Python GPU stack needed
└──────────┬───────────┘
           │  HTTP API /api/generate
           ▼
┌──────────────────────┐
│  RepairWise Engine   │  engine.py
│  Safety Triage       │  Deterministic — always runs first
│  4-Engine RAG        │  TF-IDF + sentence-transformers
│  Ollama backend      │  REPAIRWISE_BACKEND=ollama
└──────────┬───────────┘
           │
           ▼
  Structured answer (6 languages)
  Diagnosis · Risk · Steps · Sources
```

### Running locally (anyone can do this)
```bash
# 1. Install Ollama
curl -fsSL https://ollama.ai/install.sh | sh

# 2. Pull Gemma 4
ollama pull gemma4:e2b

# 3. Clone RepairWise
git clone https://github.com/SocAbdul/repairwise-gemma
cd repairwise-gemma
pip install -r requirements.txt

# 4. Run with Ollama backend
REPAIRWISE_BACKEND=ollama python app.py
```

No HuggingFace token. No GPU required. Works on any laptop with 8GB RAM.
